# Hotel Booking Demand - Data Analysis & Preprocessing

**VoidMain Technologies AI Internship - Task 2**

This notebook is the walkthrough of the whole preprocessing project. All logic lives in `src/`; the notebook calls those functions and explains the results, so the notebook and `python main.py` always agree.

**Dataset:** Hotel booking demand (Antonio, Almeida & Nunes, 2019, *Data in Brief*) - real bookings from a resort hotel and a city hotel in Portugal, 2015-2017.  
**Target (context only):** `is_canceled`

## 0. Setup

In [1]:
import sys
from pathlib import Path

# Make the project root importable when running from notebooks/.
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src import config
from src.download_data import ensure_raw_data
from src.load_data import column_overview, data_quality_snapshot, load_raw_data, missing_summary

config.setup_logging()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

## 1. Data understanding

### 1.1 Load the raw data
`ensure_raw_data()` downloads the CSV if it isn't there yet. `load_raw_data()` reads it with `na_values=['NULL']`, because this mirror stores missing `agent` / `company` / `country` values as the text `"NULL"`.

In [2]:
df = load_raw_data(ensure_raw_data())
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

12:42:34 | INFO    | src.download_data  | Raw data found at data\raw\hotel_bookings.csv - skipping download


12:42:34 | INFO    | src.load_data      | Loaded 119,390 rows x 32 columns from hotel_bookings.csv


Shape: 119,390 rows x 32 columns


### 1.2 First rows

In [3]:
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


### 1.3 Data types and memory (`info`)

In [4]:
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

### 1.4 Column overview
One row per column: type, missing count/%, number of unique values and an example.

In [5]:
column_overview(df)

,dtype,non_null,missing,missing_pct,n_unique,example
hotel,str,119390,0,0.00,2,Resort Hotel
is_canceled,int64,119390,0,0.00,2,0
lead_time,int64,119390,0,0.00,479,342
arrival_date_year,int64,119390,0,0.00,3,2015
arrival_date_month,str,119390,0,0.00,12,July
arrival_date_week_number,int64,119390,0,0.00,53,27
arrival_date_day_of_month,int64,119390,0,0.00,31,1
stays_in_weekend_nights,int64,119390,0,0.00,17,0
stays_in_week_nights,int64,119390,0,0.00,35,0
adults,int64,119390,0,0.00,14,2


### 1.5 Numeric summary (`describe`)

In [6]:
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
is_canceled,119390.0,0.37,0.48,0.00,0.00,0.00,1.0,1.0
lead_time,119390.0,104.01,106.86,0.00,18.00,69.00,160.0,737.0
arrival_date_year,119390.0,2016.16,0.71,2015.00,2016.00,2016.00,2017.0,2017.0
arrival_date_week_number,119390.0,27.17,13.61,1.00,16.00,28.00,38.0,53.0
arrival_date_day_of_month,119390.0,15.80,8.78,1.00,8.00,16.00,23.0,31.0
stays_in_weekend_nights,119390.0,0.93,1.00,0.00,0.00,1.00,2.0,19.0
stays_in_week_nights,119390.0,2.50,1.91,0.00,1.00,2.00,3.0,50.0
adults,119390.0,1.86,0.58,0.00,2.00,2.00,2.0,55.0
children,119386.0,0.10,0.40,0.00,0.00,0.00,0.0,10.0
babies,119390.0,0.01,0.10,0.00,0.00,0.00,0.0,10.0


### 1.6 Categorical summary

In [7]:
df.select_dtypes(exclude="number").describe().T

,count,unique,top,freq
hotel,119390,2,City Hotel,79330
arrival_date_month,119390,12,August,13877
meal,119390,5,BB,92310
country,118902,177,PRT,48590
market_segment,119390,8,Online TA,56477
distribution_channel,119390,5,TA/TO,97870
reserved_room_type,119390,10,A,85994
assigned_room_type,119390,12,A,74053
deposit_type,119390,3,No Deposit,104641
customer_type,119390,4,Transient,89613


### 1.7 Missing values

In [8]:
missing_summary(df)

,missing,missing_pct
company,112593,94.31
agent,16340,13.69
country,488,0.41
children,4,0.00


### 1.8 Target balance

In [9]:
target_counts = df[config.TARGET].value_counts().rename({0: "not canceled", 1: "canceled"})
pd.DataFrame({"count": target_counts, "pct": (target_counts / len(df) * 100).round(2)})

,count,pct
is_canceled,,
not canceled,75166,62.96
canceled,44224,37.04


### 1.9 Cancellation rate by hotel

In [10]:
df.groupby("hotel")[config.TARGET].agg(bookings="size", cancel_rate="mean").assign(
    cancel_rate=lambda t: (t["cancel_rate"] * 100).round(2)
)

,bookings,cancel_rate
hotel,,
City Hotel,79330,41.73
Resort Hotel,40060,27.76


### 1.10 Data-quality snapshot
Counting the known problems *before* fixing anything in Stage 2.

In [11]:
pd.Series(data_quality_snapshot(df), name="value").to_frame()

,value
rows,119390.00
duplicate_rows,31994.00
rows_with_any_missing,119173.00
zero_guest_bookings,180.00
negative_adr,1.00
zero_adr,1959.00
adr_max,5400.00
cancellation_rate_pct,37.04


### 1.11 Closer look at specific columns

In [12]:
print('Hidden "Undefined" categories:')
for col in ["meal", "market_segment", "distribution_channel"]:
    print(f"  {col:22s}", (df[col] == "Undefined").sum())

print("\nADR extremes:")
print(df["adr"].describe(percentiles=[0.01, 0.5, 0.99]).round(2).to_string())

print("\nTop 10 countries (of", df["country"].nunique(), "unique):")
print((df["country"].value_counts(normalize=True).head(10) * 100).round(2).to_string())

Hidden "Undefined" categories:
  meal                   1169
  market_segment         2
  distribution_channel   5

ADR extremes:
count    119390.00
mean        101.83
std          50.54
min          -6.38
1%            0.00
50%          94.58
99%         252.00
max        5400.00

Top 10 countries (of 177 unique):
country
PRT    40.87
GBR    10.20
FRA     8.76
ESP     7.21
DEU     6.13
ITA     3.17
IRL     2.84
BEL     1.97
BRA     1.87
NLD     1.77


In [13]:
print("reservation_status vs is_canceled (leakage check):")
pd.crosstab(df["reservation_status"], df[config.TARGET])

reservation_status vs is_canceled (leakage check):


is_canceled,0,1
reservation_status,,
Canceled,0,43017
Check-Out,75166,0
No-Show,0,1207


### 1.12 Key observations (Stage 1)

**Size & types**
- 119,390 bookings x 32 columns, 2015-2017. City Hotel = 79,330 bookings, Resort Hotel = 40,060.
- 16 int, 4 float, 12 text columns. `agent` and `company` are numeric **IDs**, not quantities -> must be treated as categories.
- `children` is float only because it has missing values; it should be an integer count.
- Arrival date is split across 4 columns (year, month name, week number, day) -> good material for feature engineering.

**Missing values** (after treating `"NULL"` as missing)
| Column | Missing | % | Likely meaning |
|---|---|---|---|
| `company` | 112,593 | 94.3% | No company involved (not a data error) |
| `agent` | 16,340 | 13.7% | Booked directly, no travel agent |
| `country` | 488 | 0.4% | Genuinely unknown |
| `children` | 4 | ~0% | Genuinely unknown |

So the large gaps are *structural* ("not applicable"), not random - which drives the Stage 2 treatment.

**Duplicates:** 31,994 fully identical rows (26.8%). The dataset has no booking ID, so some may be genuine identical group bookings - needs a reasoned decision in Stage 2.

**Invalid values / outliers**
- 180 bookings with 0 adults + 0 children + 0 babies -> invalid.
- `adr` (average daily rate): 1 negative value (-6.38), 1,959 zeros (complimentary stays exist), max 5,400 while the 99th percentile is only 252 -> extreme outlier.
- Suspicious maxima: `adults` = 55, `babies` = 10, `required_car_parking_spaces` = 8.

**Hidden missing values:** `meal` has 1,169 `"Undefined"` (per the dataset paper, Undefined = SC / no meal); `market_segment` has 2 and `distribution_channel` 5 `"Undefined"`.

**High cardinality:** `country` has 177 values, but PRT (40.9%), GBR, FRA, ESP, DEU cover ~73% -> rare countries can be grouped.

**Target:** 37.0% canceled - moderately imbalanced. City Hotel cancels much more (41.7%) than Resort (27.8%).

**Leakage confirmed:** `reservation_status` maps perfectly onto the target (Canceled/No-Show = canceled, Check-Out = not canceled), and `reservation_status_date` is recorded *after* the outcome. Both must be dropped from the model-ready output.